# 09. Cost Frontier

This notebook maps faithfulness against cost across the main experiment conditions.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from notebooks.notebook_utils import *

set_plot_style()
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

phase1_raw, phase1_summary = load_phase1_results()
phase2, phase2_summary = load_phase2_results()
phase1_summary['label'] = 'Phase 1 | ' + phase1_summary['architecture']
phase2_summary['label'] = phase2_summary['model_family'] + ' | ' + phase2_summary['architecture']
combined = pd.concat([
    phase1_summary[['label', 'mean_faithfulness', 'mean_total_cost_usd', 'total_cost_usd', 'mean_input_tokens']].assign(experiment='Phase 1'),
    phase2_summary[['label', 'mean_faithfulness', 'mean_total_cost_usd', 'total_cost_usd', 'mean_input_tokens']].assign(experiment='Phase 2'),
], ignore_index=True)
short_map = {
    'Phase 1 | Simple RAG': 'P1-SR',
    'Phase 1 | Advanced RAG': 'P1-AR',
    'Phase 1 | Long Context': 'P1-LC',
    'Gemini 2.5 Flash | Simple RAG': 'G25-SR',
    'Gemini 2.5 Flash | Advanced RAG': 'G25-AR',
    'Gemini 2.5 Flash | Long Context': 'G25-LC',
    'GPT-4o Mini | Simple RAG': 'G4O-SR',
    'GPT-4o Mini | Advanced RAG': 'G4O-AR',
    'GPT-4o Mini | Long Context': 'G4O-LC',
}
combined['short_label'] = combined['label'].map(short_map)
markdown_df(combined[['short_label', 'label', 'mean_faithfulness', 'mean_total_cost_usd']].sort_values('mean_faithfulness', ascending=False))

## Pareto-style scatter

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=combined, x='mean_total_cost_usd', y='mean_faithfulness', hue='experiment', style='experiment', s=150)
offsets = [(8, 8), (8, -12), (-10, 8), (-10, -12), (12, 0), (-14, 0), (0, 12), (0, -16), (14, 12)]
for idx, (_, row) in enumerate(combined.iterrows()):
    dx, dy = offsets[idx % len(offsets)]
    plt.annotate(row['short_label'],
                 (row['mean_total_cost_usd'], row['mean_faithfulness']),
                 textcoords='offset points',
                 xytext=(dx, dy),
                 ha='center',
                 va='center',
                 fontsize=9,
                 fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.75))
plt.xscale('log')
plt.title('Faithfulness vs mean cost per query')
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.tight_layout()

## Projected cost per 1,000 queries

In [ ]:
combined['cost_per_1000_queries'] = combined['mean_total_cost_usd'] * 1000
markdown_df(combined[['label', 'cost_per_1000_queries', 'mean_faithfulness']].sort_values('cost_per_1000_queries'))